# Submit SCF energy calculation

In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [ ]:
# General imports.
import urllib.parse as urlparse

import ipywidgets as ipw
import numpy as np
from IPython.display import clear_output

# AiiDA imports.
%load_ext aiida
%aiida
# AiiDAlab imports.
import aiidalab_widgets_base as awb
import aiidalab_widgets_empa as awe
from aiida import orm, plugins

from surfaces_tools.utils import wfn

# Custom imports.
from surfaces_tools.widgets import editors, inputs


In [ ]:
Cp2kScfWorkChain = plugins.WorkflowFactory("nanotech_empa.cp2k.scf")

In [ ]:
# Structure selector.
build_slab = editors.BuildSlab(title="Build slab")
input_details = inputs.InputDetails()

structure_selector = awb.StructureManagerWidget(
    importers=[
        awb.StructureUploadWidget(title="Import from computer"),
        awb.StructureBrowserWidget(title="AiiDA database"),
        awb.SmilesWidget(title="From SMILES"),
        awe.CdxmlUploadWidget(title="CDXML"),
    ],
    editors=[
        awb.BasicStructureEditor(title="Edit structure"),
        build_slab,
        awb.BasicCellEditor(),
        editors.InsertStructureWidget(title="Insert molecule"),
    ],
    storable=True,
    node_class="StructureData",
)
ipw.dlink((structure_selector, "structure"), (build_slab, "molecule"))
ipw.dlink((structure_selector, "structure"), (input_details, "structure"))
ipw.dlink((structure_selector, "structure_node"), (input_details, "structure_node"))
ipw.dlink((input_details, "details"), (build_slab, "details"))
display(structure_selector)

# Code.
code_input_widget = awb.ComputationalResourcesWidget(
    description="CP2K code:", default_calc_job_plugin="cp2k"
)
sparse_overlap_code = awb.ComputationalResourcesWidget(
    description="Overlap converter:",
    default_calc_job_plugin="nanotech_empa.sparse_overlap",
)
unfolding_code = awb.ComputationalResourcesWidget(
    description="Unfolding code:",
    default_calc_job_plugin="nanotech_empa.cp2k_unfolding",
)
bader_code = awb.ComputationalResourcesWidget(
    description="Bader code:",
    default_calc_job_plugin="nanotech_empa.bader",
)
resources = awe.ProcessResourcesWidget()

output = ipw.Output()


In [ ]:
url = urlparse.urlsplit(jupyter_notebook_url)
parsed_url = urlparse.parse_qs(url.query)
if "structure_uuid" in parsed_url:
    structure_selector.input_structure = load_node(parsed_url["structure_uuid"][0])

In [ ]:
# Protocol.
protocol = ipw.Dropdown(
    value="standard",
    options=[
        ("Standard", "standard"),
        ("Low accuracy", "low_accuracy"),
        ("Debug", "debug"),
    ],
    description="Protocol:",
    style={"description_width": "initial"},
)

write_overlap_matrix = ipw.Checkbox(
    value=False,
    description="Write overlap matrix",
    style={"description_width": "initial"},
)

compute_unfolding = ipw.Checkbox(
    value=False,
    description="Compute band unfolding",
    style={"description_width": "initial"},
)

compute_bader_charges = ipw.Checkbox(
    value=False,
    description="Compute Bader charges",
    style={"description_width": "initial"},
)

bader_cutoff = ipw.FloatText(
    value=1200.0,
    description="Bader cutoff:",
    style={"description_width": "initial"},
    layout={"width": "240px"},
)

diagonalisation_smearing = inputs.DiagonalisationSmearingWidget()

added_mos = ipw.BoundedIntText(
    value=100,
    min=0,
    max=100000,
    description="ADDED_MOS:",
    style={"description_width": "initial"},
    layout={"width": "220px"},
)

overlap_ndigits = ipw.BoundedIntText(
    value=14,
    min=1,
    max=30,
    description="Overlap digits:",
    style={"description_width": "initial"},
    layout={"width": "220px"},
)

retrieve_sparse_overlap = ipw.Checkbox(
    value=False,
    description="Also retrieve sparse overlap matrix",
    style={"description_width": "initial"},
)

overlap_threshold = ipw.FloatText(
    value=1.0e-10,
    description="Sparse threshold:",
    style={"description_width": "initial"},
    layout={"width": "240px"},
)

unfolding_a1 = ipw.Text(description="a1:", style={"description_width": "initial"}, layout={"width": "70%"})
unfolding_a2 = ipw.Text(description="a2:", style={"description_width": "initial"}, layout={"width": "70%"})
unfolding_a3 = ipw.Text(description="a3:", placeholder="Leave empty for 2D; leave a2/a3 empty for 1D", style={"description_width": "initial"}, layout={"width": "70%"})
unfolding_lattice_type = ipw.Dropdown(
    value="auto",
    options=["auto", "1d", "square", "rectangular", "hexagonal", "oblique"],
    description="Path type:",
    style={"description_width": "initial"},
)
unfolding_path = ipw.Text(value="G-K-M-G", description="Path:", style={"description_width": "initial"}, layout={"width": "260px"})
unfolding_preview = ipw.Textarea(value="", description="snapped:", disabled=True, rows=5, style={"description_width": "initial"}, layout={"width": "70%"})

sparse_retrieve_options = ipw.VBox([])
overlap_options = ipw.VBox([])
unfolding_options = ipw.VBox([])
bader_options = ipw.VBox([])


def _vector_to_text(vector):
    return " ".join(f"{x:.10g}" for x in vector)


def _parse_vector(text):
    text = text.strip()
    if not text:
        return None
    values = [float(x) for x in text.replace(",", " ").split()]
    if len(values) != 3:
        raise ValueError("Each vector must contain three numbers.")
    return np.asarray(values, dtype=float)


def _read_unfolding_vectors():
    vectors = [_parse_vector(w.value) for w in (unfolding_a1, unfolding_a2, unfolding_a3)]
    if vectors[0] is None:
        raise ValueError("a1 is required.")
    if vectors[1] is None and vectors[2] is not None:
        raise ValueError("a2 cannot be empty when a3 is set.")
    return np.asarray([v for v in vectors if v is not None], dtype=float)


def _lattice_matrix(vectors):
    dim = vectors.shape[0]
    return vectors[:dim, :dim].T


def _snap_vectors(approx_vectors, supercell_vectors):
    dim = approx_vectors.shape[0]
    mf = np.linalg.solve(_lattice_matrix(approx_vectors), _lattice_matrix(supercell_vectors))
    center = np.rint(mf).astype(int)
    best = None
    for delta in np.ndindex(*([3] * (dim * dim))):
        matrix = center + np.asarray(delta, dtype=int).reshape(dim, dim) - 1
        if abs(np.linalg.det(matrix)) < 0.5:
            continue
        primitive_lattice = _lattice_matrix(supercell_vectors) @ np.linalg.inv(matrix)
        snapped = np.zeros((dim, 3))
        snapped[:, :dim] = primitive_lattice.T
        score = float(np.linalg.norm(snapped - approx_vectors))
        item = (score, matrix, snapped, mf)
        if best is None or item[0] < best[0]:
            best = item
    if best is None:
        raise ValueError("Could not snap vectors to the supercell.")
    return best


def _path_labels_for_type(lattice_type, dim):
    if dim == 1:
        return "G, X"
    if lattice_type in ("auto", "hexagonal"):
        return "G, K, M"
    if lattice_type in ("square", "rectangular", "oblique"):
        return "G, X, S, Y, M"
    return ""


def _update_unfolding_preview(_=None):
    try:
        if not structure_selector.structure_node:
            unfolding_preview.value = "Select a structure to preview the exact tiling."
            return
        approx = _read_unfolding_vectors()
        dim = approx.shape[0]
        supercell = np.asarray(structure_selector.structure_node.cell, dtype=float)[:dim]
        correction, matrix, snapped, mf = _snap_vectors(approx, supercell)
        labels = _path_labels_for_type(unfolding_lattice_type.value, dim)
        unfolding_preview.value = (
            "Exact primitive vectors used:\n"
            + "\n".join(_vector_to_text(row) for row in snapped)
            + f"\n\nM =\n{matrix}\n"
            + f"det(M) = {int(round(abs(np.linalg.det(matrix))))}\n"
            + f"correction = {correction:.6g} Angstrom\n"
            + f"available standard labels: {labels}"
        )
    except Exception as exc:
        unfolding_preview.value = f"Cannot preview unfolding vectors yet: {exc}"


def _update_sparse_retrieve_options(_=None):
    if retrieve_sparse_overlap.value:
        sparse_retrieve_options.children = [overlap_threshold, sparse_overlap_code]
    elif compute_unfolding.value:
        sparse_retrieve_options.children = [overlap_threshold]
    else:
        sparse_retrieve_options.children = []


def _update_unfolding_options(_=None):
    if compute_unfolding.value:
        write_overlap_matrix.value = True
    unfolding_options.children = (
        [
            ipw.HTML("<b>Primitive vectors in Angstrom</b>"),
            unfolding_a1,
            unfolding_a2,
            unfolding_a3,
            ipw.HBox([unfolding_lattice_type, unfolding_path, unfolding_code]),
            unfolding_preview,
        ]
        if compute_unfolding.value and not compute_bader_charges.value
        else []
    )
    _update_unfolding_preview()
    _update_overlap_options()


def _update_bader_options(_=None):
    if compute_bader_charges.value:
        write_overlap_matrix.value = False
        retrieve_sparse_overlap.value = False
        compute_unfolding.value = False
    write_overlap_matrix.disabled = compute_bader_charges.value or compute_unfolding.value
    compute_unfolding.disabled = compute_bader_charges.value
    bader_options.children = [bader_cutoff, bader_code] if compute_bader_charges.value else []
    _update_overlap_options()
    _update_unfolding_options()


def _update_overlap_options(_=None):
    needs_diag = (write_overlap_matrix.value or compute_unfolding.value) and not compute_bader_charges.value
    overlap_options.children = (
        [
            diagonalisation_smearing,
            added_mos,
            overlap_ndigits,
            retrieve_sparse_overlap,
            sparse_retrieve_options,
        ]
        if needs_diag
        else []
    )
    _update_sparse_retrieve_options()

for widget in (unfolding_a1, unfolding_a2, unfolding_a3, unfolding_lattice_type):
    widget.observe(_update_unfolding_preview, "value")
write_overlap_matrix.observe(_update_overlap_options, "value")
retrieve_sparse_overlap.observe(_update_sparse_retrieve_options, "value")
compute_unfolding.observe(_update_unfolding_options, "value")
compute_bader_charges.observe(_update_bader_options, "value")
_update_bader_options()

parent_calc_folder = ipw.Text(
    value="",
    description="WFN restart folder:",
    placeholder="Optional remote path on the selected computer",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)

advanced_settings = ipw.Accordion(
    children=[ipw.VBox([parent_calc_folder])],
    selected_index=None,
)
advanced_settings.set_title(0, "Advanced settings")





In [ ]:
workflow_description = ipw.Text(
    description="Workflow description:",
    placeholder="Provide the description here.",
    style={"description_width": "initial"},
    layout={"width": "70%"},
)

In [ ]:
ipw.dlink((code_input_widget, "value"), (input_details, "selected_code"))


def prepare_scf():
    with output:
        clear_output()
    if not structure_selector.structure_node:
        can_submit, msg = False, "Select a structure first."
    elif not code_input_widget.value:
        can_submit, msg = False, "Select CP2K code."
    elif compute_bader_charges.value and not bader_code.value:
        can_submit, msg = False, "Select Bader code."
    elif compute_unfolding.value and not unfolding_code.value:
        can_submit, msg = False, "Select unfolding code."
    elif compute_unfolding.value and not unfolding_a1.value.strip():
        can_submit, msg = False, "Enter at least primitive vector a1 for unfolding."
    elif (
        not compute_bader_charges.value
        and retrieve_sparse_overlap.value
        and not sparse_overlap_code.value
    ):
        can_submit, msg = False, "Select overlap converter code."
    else:
        can_submit, msg, parameters = input_details.return_final_dictionary()
        parent_calc_folder_path = parent_calc_folder.value.strip()

    if not can_submit:
        with output:
            print(msg)
            return

    code = orm.load_code(code_input_widget.value)
    dft_params_dict = parameters["dft_params"]
    dft_params_dict.setdefault("periodic", "XYZ")
    dft_params_dict["elpa_switch"] = True
    dft_params_dict["sc_diag"] = (
        False
        if compute_bader_charges.value
        else diagonalisation_smearing.enable_diagonalisation.value
    )

    if (
        not compute_bader_charges.value
        and (write_overlap_matrix.value or compute_unfolding.value)
        and added_mos.value > 0
    ):
        dft_params_dict["added_mos"] = added_mos.value

    if (
        not compute_bader_charges.value
        and (write_overlap_matrix.value or compute_unfolding.value)
        and diagonalisation_smearing.smearing_enabled
    ):
        dft_params_dict["smear_t"] = diagonalisation_smearing.smearing_temperature.value
        dft_params_dict["force_multiplicity"] = diagonalisation_smearing.force_multiplicity.value

    builder = Cp2kScfWorkChain.get_builder()
    builder.protocol = orm.Str(protocol.value)
    builder.metadata.label = (
        "CP2K_SCF_bader"
        if compute_bader_charges.value
        else "CP2K_SCF_unfolding"
        if compute_unfolding.value
        else "CP2K_SCF_overlap"
        if write_overlap_matrix.value
        else "CP2K_SCF"
    )
    builder.metadata.description = workflow_description.value
    builder.cp2k_code = code
    builder.options = {
        "max_wallclock_seconds": resources.walltime_seconds,
        "resources": {
            "num_machines": resources.nodes,
            "num_mpiprocs_per_machine": resources.tasks_per_node,
            "num_cores_per_mpiproc": resources.threads_per_task,
        },
    }
    builder.structure = structure_selector.structure_node
    builder.dft_params = orm.Dict(dft_params_dict)
    builder.compute_bader_charges = orm.Bool(compute_bader_charges.value)
    builder.write_overlap_matrix = orm.Bool(
        (write_overlap_matrix.value or compute_unfolding.value) and not compute_bader_charges.value
    )
    builder.retrieve_sparse_overlap = orm.Bool(
        retrieve_sparse_overlap.value and not compute_bader_charges.value
    )
    builder.compute_unfolding = orm.Bool(
        compute_unfolding.value and not compute_bader_charges.value
    )
    builder.overlap_ndigits = orm.Int(overlap_ndigits.value)
    builder.overlap_threshold = orm.Float(overlap_threshold.value)
    builder.bader_cutoff = orm.Float(bader_cutoff.value)
    if retrieve_sparse_overlap.value and not compute_bader_charges.value:
        builder.sparse_overlap_code = orm.load_code(sparse_overlap_code.value)
    if compute_unfolding.value and not compute_bader_charges.value:
        builder.unfolding_code = orm.load_code(unfolding_code.value)
        builder.unfolding_primitive_vectors = orm.Str(
            "; ".join(w.value.strip() for w in (unfolding_a1, unfolding_a2, unfolding_a3) if w.value.strip())
        )
        builder.unfolding_path = orm.Str(unfolding_path.value.strip() or "G-K-M-G")
        builder.unfolding_lattice_type = orm.Str(unfolding_lattice_type.value)
    if compute_bader_charges.value:
        builder.bader_code = orm.load_code(bader_code.value)

    if parent_calc_folder_path:
        selected_parent_calc_folder = orm.RemoteData(
            computer=code.computer,
            remote_path=parent_calc_folder_path,
        )
        print(
            "Restarting from selected parent folder: "
            f"{parent_calc_folder_path}"
        )
        builder.parent_calc_folder = selected_parent_calc_folder
    else:
        # Check if a restart wfn is available.
        wave_function = None
        if structure_selector.structure_node.is_stored:
            wave_function = wfn.structure_available_wfn(
                node=structure_selector.structure_node,
                relative_replica_id=None,
                current_hostname=code.computer.hostname,
                return_path=False,
                dft_params=dft_params_dict,
            )
        if wave_function is not None:
            print(f"Restarting from wfn in folder: {wave_function.pk}")
            builder.parent_calc_folder = wave_function

    return builder



In [ ]:
btn_submit = awb.SubmitButtonWidget(
    Cp2kScfWorkChain,
    inputs_generator=prepare_scf,
    disable_after_submit=False,
    append_output=True,
)

In [ ]:
# Resources estimation.
resources_estimation = awe.ResourcesEstimatorWidget(
    price_link="https://2go.cscs.ch/offering/swiss_academia/institutional_customers/",
    price_per_hour=2.85,
)
resources_estimation.link_to_resources_widget(resources)
ipw.dlink((input_details, "details"), (resources_estimation, "details"))
ipw.dlink((input_details, "uks"), (resources_estimation, "uks"))
_ = ipw.dlink((code_input_widget, "value"), (resources_estimation, "selected_code"))

# Inputs

In [ ]:
display(
    input_details,
    protocol,
    compute_bader_charges,
    bader_options,
    compute_unfolding,
    unfolding_options,
    write_overlap_matrix,
    overlap_options,
    advanced_settings,
)


# Code and resources

In [ ]:
display(code_input_widget, ipw.HBox([resources, resources_estimation]))

# Submit

In [ ]:
display(workflow_description, btn_submit, output)